# Summative Lab: DataVine Analytics
### Junior Data Scientist Prototype — Wine Classification, Feed Recommendation, Crime Clustering

This notebook covers three client projects for DataVine Analytics:

1. **Wine Classification System** — k-NN + PCA + GridSearchCV hyperparameter tuning
2. **Agricultural Feed Recommendation Engine** — PCA + cosine similarity on Chickwts
3. **Regional Crime Pattern Analysis** — Feature selection + PCA + K-Means/GMM clustering on USArrests

## 0. Setup

In [1]:
# Install rdatasets (gives Python access to classic R datasets: chickwts, USArrests)
! pip install rdatasets -q

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.datasets import load_wine
from rdatasets import data

sns.set_style("whitegrid")

## 1. Dataset Preparation
Load all three datasets, inspect for missing values, and confirm structure before modelling.

In [ ]:
# --- Load Datasets ---

# Dataset 1: Wine (for k-NN classification)
wine_data = load_wine()
wine_df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
wine_df["target"] = wine_data.target   # already numeric: 0, 1, 2 (three wine cultivars)

# Dataset 2: Chickwts (for Recommendation System)
chickwts = data("chickwts").reset_index(drop=True)

# Dataset 3: USArrests (for Clustering)
usarrests = data("USArrests").reset_index()
usarrests = usarrests.rename(columns={"index": "State"})

In [ ]:
# --- Inspect for missing values ---
print("Missing values — Wine:\n", wine_df.isnull().sum(), "\n")
print("Missing values — Chickwts:\n", chickwts.isnull().sum(), "\n")
print("Missing values — USArrests:\n", usarrests.isnull().sum(), "\n")

# Drop any rows with missing values (defensive check — these datasets are clean by default)
wine_df = wine_df.dropna().reset_index(drop=True)
chickwts = chickwts.dropna().reset_index(drop=True)
usarrests = usarrests.dropna().reset_index(drop=True)

In [ ]:
# --- Dataset summaries ---
print("Wine Dataset:", wine_df.shape)
display(wine_df.head())

print("\nChickwts Dataset:", chickwts.shape)
display(chickwts.head())

print("\nUSArrests Dataset:", usarrests.shape)
display(usarrests.head())

## 2. Wine Classification System (k-NN + PCA)
**Goal:** classify wine cultivars from their chemical properties.

Steps: standardize features → PCA (retain 95% variance) → GridSearchCV over `k` and distance metric → train and evaluate.

In [ ]:
# Features and target (target is already numeric — no encoding needed)
X = wine_df.drop(columns=["target"])
y = wine_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA — retain 95% variance
pca_wine = PCA(n_components=0.95)
X_train_pca = pca_wine.fit_transform(X_train_scaled)
X_test_pca = pca_wine.transform(X_test_scaled)
print(f"Components retained for 95% variance: {pca_wine.n_components_} (from {X.shape[1]} original features)")

In [ ]:
# Hyperparameter tuning: optimal k and distance metric
param_grid = {"n_neighbors": range(1, 21), "metric": ["euclidean", "manhattan", "minkowski"]}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring="accuracy")
grid_search.fit(X_train_pca, y_train)

best_k_knn = grid_search.best_params_["n_neighbors"]
best_metric_knn = grid_search.best_params_["metric"]
print(f"Best k: {best_k_knn}, Best distance metric: {best_metric_knn}")

# Train the best k-NN model
knn = KNeighborsClassifier(n_neighbors=best_k_knn, metric=best_metric_knn)
knn.fit(X_train_pca, y_train)

y_pred = knn.predict(X_test_pca)
knn_accuracy = accuracy_score(y_test, y_pred)

print("\nk-NN Classification Report:\n", classification_report(y_test, y_pred))
print(f"k-NN Accuracy: {knn_accuracy:.2f}")

## 3. Agricultural Feed Recommendation Engine (PCA + Cosine Similarity)
**Goal:** recommend feed types with similar performance (chick weight outcomes) to a given feed.

Steps: standardize `weight` → PCA to 1 component → cosine similarity matrix → recommend top-N *distinct* similar feeds.

In [ ]:
# Standardize weight data
scaler = StandardScaler()
chickwts_scaled = scaler.fit_transform(chickwts[["weight"]])

# Apply PCA (reduce to 1 principal component)
pca_chick = PCA(n_components=1)
chickwts_pca = pca_chick.fit_transform(chickwts_scaled)

# Compute similarity matrix
similarity_matrix = cosine_similarity(chickwts_pca)

In [ ]:
def recommend_feeds(feed_name, num_recommendations=3):
    """Recommend distinct feed types most similar to feed_name, excluding itself."""
    matches = chickwts.index[chickwts["feed"] == feed_name].tolist()
    if not matches:
        return f"Feed '{feed_name}' not found in dataset."
    feed_index = matches[0]

    similarity_scores = list(enumerate(similarity_matrix[feed_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    recommended = []
    for i, score in similarity_scores:
        candidate = chickwts.iloc[i]["feed"]
        if i != feed_index and candidate not in recommended:
            recommended.append(candidate)
        if len(recommended) == num_recommendations:
            break
    return recommended

# Example recommendations
for feed in ["soybean", "casein", "linseed"]:
    print(f"Recommended feeds for '{feed}':", recommend_feeds(feed))

## 4. Regional Crime Pattern Analysis (Feature Selection + PCA + K-Means/GMM)
**Goal:** discover natural groupings of US states by crime statistics.

Steps: standardize → select top-3 features → PCA to 2 components → find best `k` via elbow (K-Means) and BIC (GMM) → cluster → visualize.

*Note:* `SelectKBest` needs a scoring target. Since this task is unsupervised, we score each feature's relationship to a **violent-crime index** (Murder + Assault + Rape) using `f_regression`, then keep the top 3 — a reasonable proxy target for feature relevance in a clustering context.

In [ ]:
numeric_cols = ["Murder", "Assault", "UrbanPop", "Rape"]

# Standardize
scaler = StandardScaler()
usarrests_scaled = scaler.fit_transform(usarrests[numeric_cols])

# Feature selection: keep top 3 features
crime_index = usarrests[["Murder", "Assault", "Rape"]].sum(axis=1)
selector = SelectKBest(score_func=f_regression, k=3)
usarrests_selected = selector.fit_transform(usarrests_scaled, crime_index)
selected_features = [numeric_cols[i] for i in selector.get_support(indices=True)]
print("Top 3 selected features:", selected_features)

# Apply PCA — reduce to 2 principal components
pca_arrests = PCA(n_components=2)
usarrests_pca = pca_arrests.fit_transform(usarrests_selected)
print(f"Explained variance (2 PCs): {pca_arrests.explained_variance_ratio_.sum():.2%}")

In [ ]:
# --- K-Means: find best k via elbow method (inertia) ---
inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(usarrests_pca)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertia, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.show()

In [ ]:
# Pick best_k_kmeans by inspecting the elbow plot above.
# Programmatic heuristic: the k after which inertia's rate of decrease flattens most.
diffs = np.diff(inertia)
second_diffs = np.diff(diffs)
best_k_kmeans = int(np.argmax(second_diffs) + 2)  # +2 to offset the double-diff index shift
print(f"Best number of clusters for K-Means (elbow heuristic): {best_k_kmeans}")

kmeans = KMeans(n_clusters=best_k_kmeans, random_state=42, n_init=10)
usarrests["KMeans_Cluster"] = kmeans.fit_predict(usarrests_pca)

In [ ]:
# --- GMM: find best number of components via BIC ---
best_k_gmm = 1
lowest_bic = np.inf
bic_scores = []

for k in range(1, 11):
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(usarrests_pca)
    bic = gmm.bic(usarrests_pca)
    bic_scores.append(bic)
    if bic < lowest_bic:
        lowest_bic = bic
        best_k_gmm = k

print(f"Best number of clusters for GMM (BIC): {best_k_gmm}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, 11), bic_scores, marker="o", color="green")
plt.xlabel("Number of components")
plt.ylabel("BIC")
plt.title("BIC Scores for GMM")
plt.axvline(best_k_gmm, color="red", linestyle="--", label=f"chosen k={best_k_gmm}")
plt.legend()
plt.show()

gmm = GaussianMixture(n_components=best_k_gmm, random_state=42)
usarrests["GMM_Cluster"] = gmm.fit_predict(usarrests_pca)

In [ ]:
# --- Visualize and compare clustering results ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(x=usarrests_pca[:, 0], y=usarrests_pca[:, 1],
                 hue=usarrests["KMeans_Cluster"], palette="viridis", ax=axes[0])
axes[0].set_title("K-Means Clustering on USArrests")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")

sns.scatterplot(x=usarrests_pca[:, 0], y=usarrests_pca[:, 1],
                 hue=usarrests["GMM_Cluster"], palette="viridis", ax=axes[1])
axes[1].set_title("GMM Clustering on USArrests")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.show()

## 5. Model Evaluation and Interpretation — Summary

In [ ]:
print("="*65)
print("SUMMARY OF RESULTS — DataVine Analytics Prototypes")
print("="*65)

print("\n1. Wine Classification System (k-NN)")
print(f"   PCA components used: {pca_wine.n_components_}")
print(f"   Best k: {best_k_knn} | Best distance metric: {best_metric_knn}")
print(f"   Test accuracy: {knn_accuracy:.2%}")

print("\n2. Feed Recommendation Engine (PCA + Cosine Similarity)")
print(f"   Recommended feeds for 'soybean': {recommend_feeds('soybean')}")

print("\n3. Regional Crime Pattern Analysis (K-Means & GMM)")
print(f"   Features selected: {selected_features}")
print(f"   Best K-Means clusters: {best_k_kmeans}")
print(f"   Best GMM components: {best_k_gmm}")

### Interpretation notes for stakeholders

- **Wine Classification:** PCA compressed the 13 chemical measurements down to a handful of components while keeping 95% of the variance, and the tuned k-NN model separates the three cultivars with high accuracy — suitable for automated inventory/QC tagging.
- **Feed Recommendation:** the recommender surfaces feeds with statistically similar chick-weight outcomes, giving farmers substitutable options when a preferred feed is unavailable or priced out.
- **Crime Pattern Analysis:** K-Means and GMM are compared side-by-side on the same PCA-reduced feature space — K-Means gives hard groupings useful for regional policy targeting, while GMM's probabilistic assignments highlight states that sit ambiguously between crime-pattern clusters (worth a closer manual look before policy decisions).